In [12]:
import fitz  # PyMuPDF
import pandas as pd

In [13]:
pdf_path = "test_sample/Ficha_Ponto_Simplificada_André_Luis.pdf"

In [14]:
# pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
doc = fitz.open(pdf_path)
# self.timesheet_data = []

for page in doc:
    tabs = page.find_tables()

    for tab in tabs.tables:
        print(tab.to_pandas())

             Col0      Col1      Col2 SCALA TRANSPORTE E ADMINISTRACAO LTDA. 88.501.093/0001-89     Col4   Col5       Col6             Col7             Col8           Col9         Col10        Col11       Col12          Col13         Col14           Col15            Col16           Col17                                              Col18                  Col19    Col20  Col21                   Col22    Col23  Col24          Col25
0                      None      None  Ficha Ponto Simplificada Período: 21/05/2025 à...            None   None       None             None             None           None          None         None        None           None          None            None             None            None                                               None                   None     None   None                    None     None   None           None
1            None      None      None  Funcionário: ANDRE LUIS DE MORAES DA ROSA CPF:...            None   None       None          

In [52]:
import re
from pandas import DataFrame

# Convert the table to a DataFrame
df_tab = tab.to_pandas()

def reset_column_names(df: DataFrame) -> DataFrame:
    df.columns = [f"Col_{i}" for i in range(df.shape[1])]
    return df

# Define a function to check if a string starts with a date in the format DD/MM/YY
def starts_with_date(val):
    if isinstance(val, str):
        return bool(re.match(r'^\d{2}/\d{2}/\d{2}', val.strip()))
    return False

def drop_unwanted_columns(df: DataFrame) -> DataFrame:
    # Drop all columns that are fully with None
    df = df.dropna(axis=1, how='all')
    return df

# Filter rows where the first column starts with a date
df_tab = reset_column_names(df_tab)
filtered_df = df_tab[df_tab.iloc[:, 0].apply(starts_with_date)].reset_index(drop=True)
df_timesheet = drop_unwanted_columns(filtered_df)


print(df_timesheet)

           Col_0     Col_1  Col_4  Col_5  Col_7  Col_8 Col_10 Col_11 Col_13 Col_14 Col_16 Col_17 Col_19 Col_20 Col_21 Col_22 Col_23 Col_24 Col_25
0   21/05/25 qua  Trabalho  06:24  01:26  08:00  16:59  05:30  12:11  04:48  04:48  01:21  00:42  05:57  03:02  08:59                       03:02
1   22/05/25 qui  Trabalho  10:35  23:42  08:00  11:58  09:09  04:03  07:55  07:55  01:09         02:16  01:42  03:58                       01:42
2   23/05/25 sex  Trabalho  08:05  20:32  08:00  10:56  08:23  05:10  05:46  05:46  01:11  00:20  02:56         02:56                            
3   24/05/25 sáb  Trabalho  08:22  18:02  08:00  08:33  11:50  01:02  07:31  07:31  01:07         00:33         00:33                            
4   25/05/25 dom  Trabalho  07:37  19:07  08:00  10:08  13:35  05:46  04:22  04:22  01:22         02:08         02:08                            
5   26/05/25 seg  Trabalho  05:54  21:17  08:00  14:21  10:47  03:00  11:21  11:21  01:02         06:21         06:21       

In [53]:
column_mapping = {
    'Col_0': 'data',
    'Col_1': 'tipo',
    'Col_4': 'inicio',
    'Col_5': 'fim',
    'Col_7': 'jornada_normal',
    'Col_8': 'jornada_diaria',
    'Col_10': 'interjornada',
    'Col_11': 'em_direcao',
    'Col_13': 'total_parado',
    'Col_14': 'sem_direcao',
    'Col_16': 'total_refeicao',
    'Col_17': 'total_repouso',
    'Col_19': 'hora_extra_diaria_diurna',
    'Col_20': 'hora_extra_diaria_noturna',
    'Col_21': 'hora_extra_diaria_total',
    'Col_22': 'hora_extra_dom_fer_diurna',
    'Col_23': 'hora_extra_dom_fer_noturna',
    'Col_24': 'hora_extra_dom_fer_total',
    'Col_25': 'hora_noturna'
}

df_timesheet_renamed = df_timesheet.rename(columns=column_mapping)
print(df_timesheet_renamed)

            data      tipo inicio    fim jornada_normal jornada_diaria interjornada em_direcao total_parado sem_direcao total_refeicao total_repouso hora_extra_diaria_diurna hora_extra_diaria_noturna hora_extra_diaria_total hora_extra_dom_fer_diurna hora_extra_dom_fer_noturna hora_extra_dom_fer_total hora_noturna
0   21/05/25 qua  Trabalho  06:24  01:26          08:00          16:59        05:30      12:11        04:48       04:48          01:21         00:42                    05:57                     03:02                   08:59                                                                                      03:02
1   22/05/25 qui  Trabalho  10:35  23:42          08:00          11:58        09:09      04:03        07:55       07:55          01:09                                  02:16                     01:42                   03:58                                                                                      01:42
2   23/05/25 sex  Trabalho  08:05  20:32          08:00

In [54]:
# Extract information from the PDF
import re
from datetime import datetime

def extract_timesheet_info(pdf_path):
    doc = fitz.open(pdf_path)
    
    # Variables to store extracted information
    company_name = None
    employee_name = None
    period = None
    daily_data = []
    
    for page in doc:
        # Extract text from the page for company, employee, and period info
        text = page.get_text()
        
        # Extract company name (usually appears at the top)
        # Look for common patterns in Portuguese timesheets
        company_patterns = [
            r'EMPRESA[:\s]*([^\n\r]+)',
            r'RAZÃO SOCIAL[:\s]*([^\n\r]+)',
            r'EMPREGADOR[:\s]*([^\n\r]+)'
        ]
        
        for pattern in company_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match and not company_name:
                company_name = match.group(1).strip()
                break
        
        # Extract employee name
        employee_patterns = [
            r'EMPREGADO[:\s]*([^\n\r]+)',
            r'FUNCIONÁRIO[:\s]*([^\n\r]+)',
            r'NOME[:\s]*([^\n\r]+)'
        ]
        
        for pattern in employee_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match and not employee_name:
                employee_name = match.group(1).strip() if match.group(1) else match.group(0).strip()
                break
        
        # Extract period
        period_patterns = [
            r'PERÍODO[:\s]*([^\n\r]+)',
            r'COMPETÊNCIA[:\s]*([^\n\r]+)',
            r'(\d{2}/\d{4})',  # MM/YYYY format
            r'(\d{2}/\d{2}/\d{4}\s*a\s*\d{2}/\d{2}/\d{4})'  # DD/MM/YYYY a DD/MM/YYYY format
        ]
        
        for pattern in period_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match and not period:
                period = match.group(1).strip()
                break
        
        # Extract table data
        tabs = page.find_tables()
        
        for tab in tabs.tables:
            df = tab.to_pandas()
            
            # Look for tables that contain daily timesheet data
            # Common column patterns in Portuguese timesheets
            if len(df.columns) >= 3:
                # Check if this looks like a daily timesheet table
                df_text = df.astype(str).values.flatten()
                has_dates = any(re.search(r'\d{1,2}/\d{1,2}', str(cell)) for cell in df_text)
                has_times = any(re.search(r'\d{1,2}:\d{2}', str(cell)) for cell in df_text)
                
                if has_dates or has_times:
                    daily_data.append(df)
    
    doc.close()
    
    return company_name, employee_name, period, daily_data

# Extract the information
company_name, employee_name, period, daily_data_list = extract_timesheet_info(pdf_path)

print("=== EXTRACTED INFORMATION ===")
print(f"Company/Employer: {company_name}")
print(f"Employee Name: {employee_name}")
print(f"Period: {period}")
print(f"\nNumber of daily data tables found: {len(daily_data_list)}")

=== EXTRACTED INFORMATION ===
Company/Employer: None
Employee Name: Período: 21/05/2025 à 20/06/2025
Period: 21/05/2025 à 20/06/2025

Number of daily data tables found: 1


In [55]:
# Let's examine the raw text and table structure more carefully
doc = fitz.open(pdf_path)

print("=== RAW TEXT ANALYSIS ===")
for page_num, page in enumerate(doc):
    text = page.get_text()
    print(f"\n--- PAGE {page_num + 1} TEXT (first 1000 chars) ---")
    print(text[:1000])
    print("...")
    
    print(f"\n--- PAGE {page_num + 1} TABLES ---")
    tabs = page.find_tables()
    
    for i, tab in enumerate(tabs.tables):
        df = tab.to_pandas()
        print(f"\nTable {i + 1} shape: {df.shape}")
        print("Table content:")
        print(df.head(10))
        print("---")

doc.close()

=== RAW TEXT ANALYSIS ===

--- PAGE 1 TEXT (first 1000 chars) ---
SCALA TRANSPORTE E ADMINISTRACAO LTDA.
Funcionário:
Período: 21/05/2025 à 20/06/2025
88.501.093/0001-89
019.450.890-08
CPF:
Ficha Ponto Simplificada
ANDRE LUIS DE MORAES DA ROSA
Domingo/Feriado (100%)
Hora Extra Diária 50%
Jornada
Data
Tipo
Início
Fim
Jornada
Normal
Jornada
Diária
Interjornada
Em
Direção
Total
Parado
Sem
Direção
Total
Refeição
Total
Repouso
Diurna
Noturna
Total
Diurna
Noturna
Total
Hora
Noturna
 21/05/25 qua
Trabalho
06:24
01:26
08:00
16:59
05:30
12:11
04:48
04:48
01:21
00:42
05:57
03:02
08:59
03:02
 22/05/25 qui
Trabalho
10:35
23:42
08:00
11:58
09:09
04:03
07:55
07:55
01:09
02:16
01:42
03:58
01:42
 23/05/25 sex
Trabalho
08:05
20:32
08:00
10:56
08:23
05:10
05:46
05:46
01:11
00:20
02:56
02:56
 24/05/25 sáb
Trabalho
08:22
18:02
08:00
08:33
11:50
01:02
07:31
07:31
01:07
00:33
00:33
 25/05/25 dom
Trabalho
07:37
19:07
08:00
10:08
13:35
05:46
04:22
04:22
01:22
02:08
02:08
TT HS Semana
58:34
48:27
28:12
30:22
3

In [57]:
# Improved extraction function based on the table structure analysis
def extract_timesheet_data(pdf_path):
    doc = fitz.open(pdf_path)
    
    # Initialize variables
    company_name = None
    employee_name = None
    period = None
    daily_timesheet = []
    
    for page in doc:
        text = page.get_text()
        
        # Extract company name (first line)
        lines = text.split('\n')
        if lines:
            company_name = lines[0].strip()
        
        # Extract employee name and period from text
        employee_match = re.search(r'Funcionário:\s*([^\n\r]+?)(?:\s+CPF:|$)', text, re.IGNORECASE)
        if employee_match:
            employee_name = employee_match.group(1).strip()
        
        period_match = re.search(r'Período:\s*([^\n\r]+)', text, re.IGNORECASE)
        if period_match:
            period = period_match.group(1).strip()
        
        # Extract table data
        tabs = page.find_tables()
        
        for tab in tabs.tables:
            df = tab.to_pandas()
            
            # Process the main timesheet table
            if df.shape[0] > 10 and df.shape[1] > 10:  # Main table is large
                
                # Find rows that contain actual daily data (dates in format DD/MM/YY)
                date_pattern = r'\d{2}/\d{2}/\d{2}\s+\w+'  # DD/MM/YY day_name
                
                for idx, row in df.iterrows():
                    # Check if this row contains a date
                    row_text = ' '.join([str(cell) for cell in row if pd.notna(cell)])
                    
                    if re.search(date_pattern, row_text):
                        # Extract the day information
                        day_info = {
                            'data': None,
                            'tipo': None,
                            'inicio': None,
                            'fim': None,
                            'jornada_normal': None,
                            'jornada_diaria': None,
                            'interjornada': None,
                            'em_direcao': None,
                            'total_parado': None,
                            'sem_direcao': None,
                            'total_refeicao': None,
                            'total_repouso': None,
                            'hora_extra_diaria_diurna': None,
                            'hora_extra_diaria_noturna': None,
                            'hora_extra_diaria_total': None,
                            'hora_extra_dom_fer_diurna': None,
                            'hora_extra_dom_fer_noturna': None,
                            'hora_extra_dom_fer_total': None,
                            'hora_noturna': None
                        }
                        
                        # Extract data and tipo from first columns
                        if pd.notna(row.iloc[0]):
                            day_info['data'] = str(row.iloc[0]).strip()
                        if pd.notna(row.iloc[1]):
                            day_info['tipo'] = str(row.iloc[1]).strip()
                        
                        # Extract times - looking for HH:MM patterns
                        time_pattern = r'\d{1,2}:\d{2}'
                        row_values = [str(cell) for cell in row if pd.notna(cell)]
                        times = []
                        
                        for cell in row_values:
                            times.extend(re.findall(time_pattern, str(cell)))
                        
                        # Assign times to appropriate fields based on position
                        if len(times) >= 2:
                            day_info['inicio'] = times[0] if len(times) > 0 else None
                            day_info['fim'] = times[1] if len(times) > 1 else None
                        
                        if len(times) >= 4:
                            day_info['jornada_normal'] = times[2] if len(times) > 2 else None
                            day_info['jornada_diaria'] = times[3] if len(times) > 3 else None
                        
                        # Look for specific time values in known positions
                        for i, cell in enumerate(row):
                            if pd.notna(cell):
                                cell_str = str(cell).strip()
                                if re.match(time_pattern, cell_str):
                                    # Map based on approximate column positions
                                    if i >= 15 and i <= 17:  # Refeição/Repouso area
                                        if not day_info['total_refeicao']:
                                            day_info['total_refeicao'] = cell_str
                                        elif not day_info['total_repouso']:
                                            day_info['total_repouso'] = cell_str
                                    elif i >= 18 and i <= 22:  # Hora Extra area
                                        if not day_info['hora_extra_diaria_diurna']:
                                            day_info['hora_extra_diaria_diurna'] = cell_str
                                        elif not day_info['hora_extra_diaria_noturna']:
                                            day_info['hora_extra_diaria_noturna'] = cell_str
                                        elif not day_info['hora_extra_diaria_total']:
                                            day_info['hora_extra_diaria_total'] = cell_str
                                    elif i >= 23:  # Hora Noturna area
                                        if not day_info['hora_noturna']:
                                            day_info['hora_noturna'] = cell_str
                        
                        daily_timesheet.append(day_info)
    
    doc.close()
    
    # Create DataFrame from daily timesheet data
    daily_df = pd.DataFrame(daily_timesheet)
    
    return company_name, employee_name, period, daily_df

# Extract the data
company_name, employee_name, period, daily_df = extract_timesheet_data(pdf_path)

print("=== EXTRACTED INFORMATION ===")
print(f"Company/Employer: {company_name}")
print(f"Employee Name: {employee_name}")
print(f"Period: {period}")
print(f"\n=== DAILY TIMESHEET DATA ===")
print(f"Shape: {daily_df.shape}")
print("\nDaily timesheet DataFrame:")
print(daily_df)

=== EXTRACTED INFORMATION ===
Company/Employer: SCALA TRANSPORTE E ADMINISTRACAO LTDA.
Employee Name: None
Period: 21/05/2025 à 20/06/2025

=== DAILY TIMESHEET DATA ===
Shape: (31, 19)

Daily timesheet DataFrame:
            data      tipo inicio    fim jornada_normal jornada_diaria interjornada em_direcao total_parado sem_direcao total_refeicao total_repouso hora_extra_diaria_diurna hora_extra_diaria_noturna hora_extra_diaria_total hora_extra_dom_fer_diurna hora_extra_dom_fer_noturna hora_extra_dom_fer_total hora_noturna
0   21/05/25 qua  Trabalho  06:24  01:26          08:00          16:59         None       None         None        None          01:21         00:42                    05:57                     03:02                   08:59                      None                       None                     None        03:02
1   22/05/25 qui  Trabalho  10:35  23:42          08:00          11:58         None       None         None        None          01:09          None         

In [59]:
# Fix employee name extraction from the raw text
doc = fitz.open(pdf_path)
text = doc[0].get_text()
doc.close()

# Extract employee name more accurately
employee_name_match = re.search(r'ANDRE LUIS DE MORAES DA ROSA', text)
if employee_name_match:
    employee_name = employee_name_match.group(0)
else:
    # Fallback pattern
    employee_name_match = re.search(r'Funcionário:\s*\n?([A-ZÁÊÇÕ\s]+)', text)
    if employee_name_match:
        employee_name = employee_name_match.group(1).strip()

print("=== FINAL EXTRACTED INFORMATION ===")
print(f"Company/Employer: {company_name}")
print(f"Employee Name: {employee_name}")  
print(f"Period: {period}")
print(f"\n=== SUMMARY OF DAILY TIMESHEET ===")
print(f"Total days in timesheet: {len(daily_df)}")
print(f"Working days: {len(daily_df[daily_df['tipo'] == 'Trabalho'])}")
print(f"Rest days (DSR/Casa): {len(daily_df[daily_df['tipo'] == 'DSR/Casa'])}")
print(f"Holidays (Feriado): {len(daily_df[daily_df['tipo'] == 'Feriado'])}")

print(f"\n=== TIMESHEET DATAFRAME COLUMNS ===")
print(daily_df.columns.tolist())

print(f"\n=== SAMPLE OF WORKING DAYS ===")
working_days = daily_df[daily_df['tipo'] == 'Trabalho']
print(working_days[['data', 'tipo', 'inicio', 'fim', 'jornada_normal', 'jornada_diaria']])

=== FINAL EXTRACTED INFORMATION ===
Company/Employer: SCALA TRANSPORTE E ADMINISTRACAO LTDA.
Employee Name: ANDRE LUIS DE MORAES DA ROSA
Period: 21/05/2025 à 20/06/2025

=== SUMMARY OF DAILY TIMESHEET ===
Total days in timesheet: 31
Working days: 25
Rest days (DSR/Casa): 5
Holidays (Feriado): 1

=== TIMESHEET DATAFRAME COLUMNS ===
['data', 'tipo', 'inicio', 'fim', 'jornada_normal', 'jornada_diaria', 'interjornada', 'em_direcao', 'total_parado', 'sem_direcao', 'total_refeicao', 'total_repouso', 'hora_extra_diaria_diurna', 'hora_extra_diaria_noturna', 'hora_extra_diaria_total', 'hora_extra_dom_fer_diurna', 'hora_extra_dom_fer_noturna', 'hora_extra_dom_fer_total', 'hora_noturna']

=== SAMPLE OF WORKING DAYS ===
            data      tipo inicio    fim jornada_normal jornada_diaria
0   21/05/25 qua  Trabalho  06:24  01:26          08:00          16:59
1   22/05/25 qui  Trabalho  10:35  23:42          08:00          11:58
2   23/05/25 sex  Trabalho  08:05  20:32          08:00          10:5

In [60]:
# Labor Law Compliance Checks
from datetime import datetime, timedelta
import warnings

def time_to_minutes(time_str):
    """Convert time string HH:MM to minutes"""
    if pd.isna(time_str) or time_str is None or time_str == 'None':
        return 0
    try:
        hours, minutes = map(int, str(time_str).split(':'))
        return hours * 60 + minutes
    except:
        return 0

def minutes_to_time(minutes):
    """Convert minutes to HH:MM format"""
    hours = minutes // 60
    mins = minutes % 60
    return f"{hours:02d}:{mins:02d}"

def check_labor_compliance(daily_df):
    """Check labor law compliance for working days"""
    
    # Filter only working days
    working_days = daily_df[daily_df['tipo'] == 'Trabalho'].copy().reset_index(drop=True)
    
    compliance_issues = []
    
    print("=== LABOR LAW COMPLIANCE ANALYSIS ===\n")
    
    # Check 1: Daily working hours > 8 hours
    print("1. DAILY WORKING HOURS CHECK (> 8 hours)")
    print("-" * 50)
    
    for idx, row in working_days.iterrows():
        jornada_minutes = time_to_minutes(row['jornada_diaria'])
        jornada_hours = jornada_minutes / 60
        
        if jornada_minutes > 480:  # 8 hours = 480 minutes
            excess_minutes = jornada_minutes - 480
            excess_time = minutes_to_time(excess_minutes)
            print(f"⚠️  {row['data']}: {row['jornada_diaria']} ({excess_time} excess)")
            compliance_issues.append({
                'data': row['data'],
                'issue': 'Excessive daily hours',
                'details': f"Worked {row['jornada_diaria']}, excess: {excess_time}"
            })
        else:
            print(f"✅ {row['data']}: {row['jornada_diaria']}")
    
    # Check 2: Meal break >= 1 hour
    print(f"\n2. MEAL BREAK CHECK (>= 1 hour)")
    print("-" * 50)
    
    for idx, row in working_days.iterrows():
        refeicao_minutes = time_to_minutes(row['total_refeicao'])
        
        if refeicao_minutes < 60:  # Less than 1 hour
            refeicao_time = minutes_to_time(refeicao_minutes) if refeicao_minutes > 0 else "Not recorded"
            print(f"⚠️  {row['data']}: {refeicao_time}")
            compliance_issues.append({
                'data': row['data'],
                'issue': 'Insufficient meal break',
                'details': f"Meal break: {refeicao_time}, required: 01:00"
            })
        else:
            print(f"✅ {row['data']}: {row['total_refeicao']}")
    
    # Check 3: Rest period between shifts >= 11 hours
    print(f"\n3. REST PERIOD BETWEEN SHIFTS CHECK (>= 11 hours)")
    print("-" * 50)
    
    for i in range(len(working_days) - 1):
        current_day = working_days.iloc[i]
        next_day = working_days.iloc[i + 1]
        
        # Parse dates and times
        try:
            current_date_str = current_day['data'].split()[0]  # Get DD/MM/YY part
            next_date_str = next_day['data'].split()[0]
            
            current_fim = current_day['fim']
            next_inicio = next_day['inicio']
            
            if pd.notna(current_fim) and pd.notna(next_inicio):
                # Convert to datetime objects for calculation
                current_date = datetime.strptime(current_date_str, '%d/%m/%y')
                next_date = datetime.strptime(next_date_str, '%d/%m/%y')
                
                # Handle end time that goes to next day (e.g., 01:26 means 01:26 next day)
                fim_hour, fim_min = map(int, str(current_fim).split(':'))
                inicio_hour, inicio_min = map(int, str(next_inicio).split(':'))
                
                # If fim time is small (like 01:26), it likely means next day
                if fim_hour < 6:  # Assuming work doesn't normally end before 6 AM
                    fim_datetime = current_date + timedelta(days=1, hours=fim_hour, minutes=fim_min)
                else:
                    fim_datetime = current_date + timedelta(hours=fim_hour, minutes=fim_min)
                
                inicio_datetime = next_date + timedelta(hours=inicio_hour, minutes=inicio_min)
                
                # Calculate rest period
                rest_period = inicio_datetime - fim_datetime
                rest_hours = rest_period.total_seconds() / 3600
                
                if rest_hours < 11:
                    rest_time_str = f"{int(rest_hours):02d}:{int((rest_hours % 1) * 60):02d}"
                    print(f"⚠️  {current_day['data']} ({current_fim}) → {next_day['data']} ({next_inicio}): {rest_time_str}")
                    compliance_issues.append({
                        'data': f"{current_day['data']} → {next_day['data']}",
                        'issue': 'Insufficient rest period',
                        'details': f"Rest period: {rest_time_str}, required: 11:00"
                    })
                else:
                    rest_time_str = f"{int(rest_hours):02d}:{int((rest_hours % 1) * 60):02d}"
                    print(f"✅ {current_day['data']} ({current_fim}) → {next_day['data']} ({next_inicio}): {rest_time_str}")
                    
        except Exception as e:
            print(f"⚠️  Error calculating rest period between {current_day['data']} and {next_day['data']}: {str(e)}")
    
    # Summary
    print(f"\n=== COMPLIANCE SUMMARY ===")
    print(f"Total working days analyzed: {len(working_days)}")
    print(f"Total compliance issues found: {len(compliance_issues)}")
    
    if compliance_issues:
        print(f"\n=== DETAILED ISSUES ===")
        for issue in compliance_issues:
            print(f"📋 {issue['data']}: {issue['issue']}")
            print(f"   Details: {issue['details']}")
    else:
        print("🎉 No compliance issues found!")
    
    return compliance_issues

# Run the compliance check
compliance_issues = check_labor_compliance(daily_df)

=== LABOR LAW COMPLIANCE ANALYSIS ===

1. DAILY WORKING HOURS CHECK (> 8 hours)
--------------------------------------------------
⚠️  21/05/25 qua: 16:59 (08:59 excess)
⚠️  22/05/25 qui: 11:58 (03:58 excess)
⚠️  23/05/25 sex: 10:56 (02:56 excess)
⚠️  24/05/25 sáb: 08:33 (00:33 excess)
⚠️  25/05/25 dom: 10:08 (02:08 excess)
⚠️  26/05/25 seg: 14:21 (06:21 excess)
⚠️  27/05/25 ter: 10:31 (02:31 excess)
⚠️  28/05/25 qua: 14:33 (06:33 excess)
⚠️  29/05/25 qui: 08:17 (00:17 excess)
⚠️  30/05/25 sex: 12:18 (04:18 excess)
⚠️  31/05/25 sáb: 09:39 (01:39 excess)
⚠️  01/06/25 dom: 08:53 (00:53 excess)
⚠️  02/06/25 seg: 13:51 (05:51 excess)
⚠️  03/06/25 ter: 11:19 (03:19 excess)
⚠️  04/06/25 qua: 08:48 (00:48 excess)
⚠️  05/06/25 qui: 10:10 (02:10 excess)
✅ 06/06/25 sex: 01:15
⚠️  09/06/25 seg: 13:22 (05:22 excess)
⚠️  10/06/25 ter: 09:14 (01:14 excess)
⚠️  11/06/25 qua: 09:37 (01:37 excess)
⚠️  12/06/25 qui: 10:09 (02:09 excess)
⚠️  13/06/25 sex: 11:40 (03:40 excess)
⚠️  16/06/25 seg: 09:14 (01: